In [1]:
# -----------------------------
# 04_distillation.ipynb — Complete corrected distillation cell
# Paste/replace entire cell with this content.
# -----------------------------
import os
import json
import math
import random
import shutil
from pathlib import Path
from time import time
from typing import List, Dict, Any

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# ----------------- USER CONFIG -----------------
DATA_SPLITS_ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/data_splits")
TRAIN_PIPELINE_OUTPUT_ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/train_pipeline")
DISTILL_OUTPUT_ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/distillation")

STUDENT_MODEL_NAME = "distilbert-base-multilingual-cased"
TEACHER_MODEL_KEYWORD = "teacher_"
MAX_EPOCHS = 6
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WARMUP_STEPS = 50
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
NUM_WORKERS = 2

# KD params
TEMPERATURE = 2.0
ALPHA = 0.5  # weight for KD loss vs CE
TOPK_K = 3   # top-k teacher-selected chunks per file; <=0 -> use all chunks
SAVE_TOPK_INFO = True

# ----------------- reproducibility -----------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------- helpers -----------------
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def make_json_serializable(o):
    """
    Convert numpy scalars/arrays and other non-JSON types to python primitives recursively.
    """
    if isinstance(o, (str, bool, type(None))):
        return o
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (int, float)):
        return o
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, dict):
        return {make_json_serializable(k): make_json_serializable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [make_json_serializable(v) for v in o]
    try:
        return str(o)
    except Exception:
        return None

def read_label_map(lang_dir: Path) -> (Dict[str,int], Dict[int,str], Dict):
    p = lang_dir / "label_map.json"
    if not p.exists():
        raise FileNotFoundError(f"label_map.json not found at {p}. Run 01_dataset_eda and 03_train_pipeline.")
    jd = json.load(open(p, encoding="utf8"))
    label_map = {str(k): int(v) for k,v in jd.get("label_map", {}).items()}
    inv_label_map = {int(v): str(k) for k,v in jd.get("label_map", {}).items()}
    metadata = jd.get("metadata", {})
    return label_map, inv_label_map, metadata

def load_split_df(lang_dir: Path, split_name: str) -> pd.DataFrame:
    p = lang_dir / f"{split_name}.csv"
    if not p.exists():
        raise FileNotFoundError(f"Expected split at {p}")
    df = pd.read_csv(p)
    if 'label' not in df.columns and 'label_id' not in df.columns:
        raise RuntimeError(f"CSV {p} must contain 'label' or 'label_id' column.")
    if 'text' not in df.columns and 'file_path' not in df.columns:
        raise RuntimeError(f"CSV {p} must contain 'file_path' or 'text' column.")
    return df

def read_text_from_row(row) -> str:
    if 'text' in row and not pd.isna(row['text']):
        return str(row['text'])
    if 'file_path' in row and not pd.isna(row['file_path']):
        fp = Path(row['file_path'])
        if fp.exists():
            return fp.read_text(encoding="utf8", errors="ignore")
        else:
            try:
                return Path(str(row['file_path'])).read_text(encoding="utf8", errors="ignore")
            except Exception:
                return ""
    return ""

def chunk_text_with_tokenizer(tokenizer, text: str, max_len: int, stride: int):
    """
    Tokenize `text` into chunks using tokenizer overflow.
    Use truncation=True and padding='max_length' so each returned chunk is exactly max_len tokens.
    Returns list of dicts with 1-D tensors.
    """
    if not text:
        return []
    enc = tokenizer(
        text,
        truncation=True,
        padding="max_length",           # ensures uniform length across chunks
        return_overflowing_tokens=True,
        max_length=max_len,
        stride=stride,
        return_offsets_mapping=False,
        return_attention_mask=True,
        return_tensors="pt",
    )
    chunks = []
    for i in range(enc["input_ids"].size(0)):
        chunks.append({
            "input_ids": enc["input_ids"][i].clone().detach().squeeze(0),
            "attention_mask": enc["attention_mask"][i].clone().detach().squeeze(0)
        })
    return chunks

class ChunkDataset(Dataset):
    def __init__(self, flat_items: List[Dict[str, Any]]):
        # flat_items: list of dicts with keys:
        #   input_ids (tensor shape (L,)), attention_mask (L,), label_id (int), file_path (str), teacher_logits (np array) optional
        self.items = flat_items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        it = self.items[idx]
        teacher_logits = it.get("teacher_logits")  # may be None
        input_ids = it["input_ids"]
        attn = it["attention_mask"]
        # ensure tensors (1D)
        if isinstance(input_ids, np.ndarray):
            input_ids = torch.tensor(input_ids, dtype=torch.long)
        if isinstance(attn, np.ndarray):
            attn = torch.tensor(attn, dtype=torch.long)
        return {
            "input_ids": input_ids,            # (L,)
            "attention_mask": attn,            # (L,)
            "label_id": torch.tensor(int(it["label_id"]), dtype=torch.long),
            "file_path": it["file_path"],
            "teacher_logits": torch.tensor(it["teacher_logits"], dtype=torch.float32) if it.get("teacher_logits") is not None else None
        }

def collate_fn(batch):
    # batch: list of dicts with input_ids (L,), attention_mask (L,), labels, teacher_logits (maybe None)
    input_ids_list = [b['input_ids'] for b in batch]
    attention_list = [b['attention_mask'] for b in batch]
    labels = torch.tensor([int(b['label_id'].item()) for b in batch], dtype=torch.long)
    file_paths = [b['file_path'] for b in batch]

    # pad sequences
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(input_ids_list, batch_first=True, padding_value=0)
    attention_padded = torch.nn.utils.rnn.pad_sequence(attention_list, batch_first=True, padding_value=0)

    teacher_logits_list = [b['teacher_logits'] for b in batch]
    if any(t is None for t in teacher_logits_list):
        teacher_logits_tensor = None
    else:
        teacher_logits_tensor = torch.stack(teacher_logits_list, dim=0)

    return {
        "input_ids": input_ids_padded,
        "attention_mask": attention_padded,
        "labels": labels,
        "file_paths": file_paths,
        "teacher_logits": teacher_logits_tensor
    }

def aggregate_chunk_logits_to_file(pred_logits_per_chunk_files: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    agg = {}
    for fp, logits_arr in pred_logits_per_chunk_files.items():
        if logits_arr.shape[0] == 0:
            agg[fp] = np.zeros((logits_arr.shape[1],), dtype=float)
        else:
            agg[fp] = logits_arr.mean(axis=0)
    return agg

def compute_metrics_file_level(y_true: List[int], y_pred: List[int], inv_label_map):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cls_report = classification_report(y_true, y_pred, target_names=[inv_label_map[i] for i in sorted(inv_label_map.keys())], zero_division=0, output_dict=True)
    cm = confusion_matrix(y_true, y_pred).tolist()
    return {"accuracy": float(acc), "macro_f1": float(macro_f1), "classification_report": cls_report, "confusion_matrix": cm}

def find_teacher_exp_dir(lang: str) -> Path:
    lang_root = TRAIN_PIPELINE_OUTPUT_ROOT / lang
    if not lang_root.exists():
        raise FileNotFoundError(f"No training outputs found for language {lang} at {lang_root}")
    candidates = [d for d in lang_root.iterdir() if d.is_dir() and d.name.startswith(TEACHER_MODEL_KEYWORD)]
    if not candidates:
        for d in lang_root.iterdir():
            if d.is_dir() and (d / "best_model").exists():
                candidates.append(d)
    if not candidates:
        raise FileNotFoundError(f"No teacher experiment directories found under {lang_root}. Run 03_train_pipeline first.")
    return sorted(candidates)[-1]

# ----------------- MAIN -----------------
TARGET_LANGS = sorted([d.name for d in DATA_SPLITS_ROOT.iterdir() if d.is_dir()])
print("[INFO] Languages detected:", TARGET_LANGS)

for LANG in TARGET_LANGS:
    print("\n" + "="*100)
    print(f"[LANG] {LANG}")
    lang_dir = DATA_SPLITS_ROOT / LANG
    if not lang_dir.exists():
        print(f"[WARN] missing splits for {LANG} at {lang_dir}; skipping.")
        continue

    EXP_ROOT = ensure_dir(DISTILL_OUTPUT_ROOT / LANG)
    EXP_OUT = ensure_dir(EXP_ROOT / f"distil_{STUDENT_MODEL_NAME.replace('/', '_')}")
    MODEL_DIR = ensure_dir(EXP_OUT / "best_model")
    ensure_dir(EXP_OUT / "checkpoints")
    LOG_DIR = ensure_dir(EXP_OUT / "logits")
    TOPK_INFO_PATH = EXP_OUT / "topk_chunks_per_file.json"

    label_map, inv_label_map, metadata = read_label_map(lang_dir)
    num_labels = len(label_map)
    CHUNK_MAX_LEN = int(metadata.get("chunk_max_len", 256))
    CHUNK_STRIDE = int(metadata.get("chunk_stride", 64))
    print(f"[INFO] label_map: {label_map}  num_labels={num_labels}")
    print(f"[INFO] chunk params: max_len={CHUNK_MAX_LEN}, stride={CHUNK_STRIDE}")

    # find teacher artifacts
    teacher_exp_dir = find_teacher_exp_dir(LANG)
    teacher_model_dir = teacher_exp_dir / "best_model"
    teacher_logits_dir = teacher_exp_dir / "logits"
    ttrain = teacher_logits_dir / "teacher_train_chunk_logits.npz"
    tval = teacher_logits_dir / "teacher_val_chunk_logits.npz"
    if not (teacher_model_dir.exists() and (ttrain.exists() or tval.exists())):
        raise FileNotFoundError(f"Teacher artifacts missing. Ensure {teacher_model_dir} and teacher logits exist: {ttrain} and/or {tval}")

    # load teacher npz and mapping
    teacher_train_npz = np.load(str(ttrain), allow_pickle=True) if ttrain.exists() else None
    teacher_val_npz = np.load(str(tval), allow_pickle=True) if tval.exists() else None
    teacher_train_map = {k: teacher_train_npz[k] for k in teacher_train_npz.files} if teacher_train_npz is not None else {}
    teacher_val_map = {k: teacher_val_npz[k] for k in teacher_val_npz.files} if teacher_val_npz is not None else {}
    print(f"[INFO] Loaded teacher logits: train_keys={len(teacher_train_map)} val_keys={len(teacher_val_map)}")

    # load mapping file if present (non-destructive mapping created earlier)
    mapping_path = teacher_logits_dir / "file_path_key_mapping.json"
    if mapping_path.exists():
        mapping = json.load(open(mapping_path, encoding="utf8"))
        print(f"[INFO] Loaded file_path_key_mapping.json ({len(mapping)} entries) from {mapping_path}")
    else:
        mapping = None
        print(f"[INFO] No file_path_key_mapping.json found at {mapping_path}; will use exact->basename fallback")

    def find_teacher_key_for_fp(fp: str, teacher_map_keys: List[str]) -> str:
        """
        Return the teacher npz key corresponding to CSV file_path `fp`.
        Priority:
          1) mapping[fp] if mapping exists
          2) exact match (fp in teacher_map_keys)
          3) basename fallback (first teacher key with same Path(...).name)
          4) return None if not found
        """
        if fp is None:
            return None
        if mapping:
            mapped = mapping.get(str(fp))
            if mapped and mapped in teacher_map_keys:
                return mapped
        if str(fp) in teacher_map_keys:
            return str(fp)
        bname = Path(str(fp)).name
        for k in teacher_map_keys:
            if Path(k).name == bname:
                return k
        return None

    # load splits
    train_df = load_split_df(lang_dir, "train")
    val_df = load_split_df(lang_dir, "val")
    test_df = load_split_df(lang_dir, "test")

    # student tokenizer
    student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME, use_fast=True)

    # prepare entries and attach precomputed teacher logits for the selected chunks
    def prepare_entries_with_teacher_topk_and_teacher_logits(df: pd.DataFrame, teacher_map: Dict[str,np.ndarray], use_topk: int, teacher_map_keys: List[str]):
        entries = []
        topk_info = {}
        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Preparing entries"):
            fp = str(row.get("file_path", "")) if 'file_path' in row else None
            text = read_text_from_row(row)
            chunks = chunk_text_with_tokenizer(student_tokenizer, text, max_len=CHUNK_MAX_LEN, stride=CHUNK_STRIDE)
            # safety: ensure chunk lengths <= CHUNK_MAX_LEN
            for ch in chunks:
                if ch['input_ids'].numel() > CHUNK_MAX_LEN:
                    ch['input_ids'] = ch['input_ids'][:CHUNK_MAX_LEN]
                    ch['attention_mask'] = ch['attention_mask'][:CHUNK_MAX_LEN]

            # find teacher logits array for this file
            teacher_arr = None
            if teacher_map:
                key = find_teacher_key_for_fp(fp, teacher_map_keys)
                if key:
                    teacher_arr = teacher_map.get(key)
            if teacher_arr is None:
                # teacher logits missing for this file: create zeros with length = n_chunks
                teacher_arr = np.zeros((len(chunks), num_labels), dtype=float)
                # warn occasionally
                if len(chunks) > 0:
                    if random.random() < 0.15:
                        print(f"[WARN] Teacher logits not found for {fp}. Using zeros for alignment.")
            # align lengths (safeguard)
            if len(chunks) != teacher_arr.shape[0]:
                if teacher_arr.shape[0] > len(chunks):
                    teacher_arr = teacher_arr[:len(chunks)]
                elif teacher_arr.shape[0] < len(chunks):
                    pad = np.zeros((len(chunks) - teacher_arr.shape[0], num_labels), dtype=float)
                    teacher_arr = np.vstack([teacher_arr, pad])

            # compute teacher softmax confidence per chunk and pick top-K indices
            if teacher_arr.size:
                soft = np.exp(teacher_arr - np.max(teacher_arr, axis=1, keepdims=True))
                soft = soft / (soft.sum(axis=1, keepdims=True) + 1e-12)
                chunk_conf = soft.max(axis=1)
            else:
                chunk_conf = np.array([])

            K = use_topk if (use_topk and use_topk > 0) else len(chunks)
            if K >= len(chunks):
                selected_idx = list(range(len(chunks)))
            else:
                if chunk_conf.size:
                    selected_idx = list(np.argsort(-chunk_conf)[:K])
                    selected_idx = sorted(selected_idx)
                else:
                    selected_idx = list(range(min(K, len(chunks))))

            sel_chunks = [chunks[i] for i in selected_idx] if len(chunks) else []
            sel_teacher_logits = teacher_arr[selected_idx] if len(selected_idx) > 0 else np.zeros((0, num_labels), dtype=float)

            if 'label_id' in row and not pd.isna(row['label_id']):
                label_id = int(row['label_id'])
            else:
                label_id = int(label_map.get(str(row['label']), -1))

            entries.append({"file_path": str(fp), "label_id": label_id, "chunks": sel_chunks, "teacher_logits": sel_teacher_logits})
            topk_info[str(fp)] = {"selected_chunk_idx": [int(x) for x in selected_idx], "n_chunks_total": int(len(chunks)), "teacher_confidence": [float(x) for x in (chunk_conf.tolist() if chunk_conf.size else [])]}
        return entries, topk_info

    print("[PREP] preparing train entries (top-k + aligned teacher logits)")
    teacher_train_keys = list(teacher_train_map.keys())
    teacher_val_keys = list(teacher_val_map.keys())
    train_entries, train_topk = prepare_entries_with_teacher_topk_and_teacher_logits(train_df, teacher_train_map, TOPK_K, teacher_train_keys)
    print("[PREP] preparing val entries (top-k + aligned teacher logits)")
    val_entries, val_topk = prepare_entries_with_teacher_topk_and_teacher_logits(val_df, teacher_val_map, TOPK_K, teacher_val_keys)
    print("[PREP] preparing test entries (all chunks; teacher logits may be missing for test)")
    test_entries, test_topk = prepare_entries_with_teacher_topk_and_teacher_logits(test_df, teacher_val_map, 0, teacher_val_keys)

    if SAVE_TOPK_INFO:
        json.dump(make_json_serializable({"train": train_topk, "val": val_topk, "test": test_topk}),
                  open(TOPK_INFO_PATH, "w", encoding="utf8"),
                  indent=2, ensure_ascii=False)
        print(f"[INFO] Saved topk info to {TOPK_INFO_PATH}")

    # flatten and preserve teacher_logits for each chunk
    flat_train = []
    for fe in train_entries:
        for ci, ch in enumerate(fe['chunks']):
            flat_train.append({
                "input_ids": ch['input_ids'],
                "attention_mask": ch['attention_mask'],
                "label_id": fe['label_id'],
                "file_path": fe['file_path'],
                "teacher_logits": fe['teacher_logits'][ci] if (fe['teacher_logits'].shape[0] > ci) else np.zeros((num_labels,), dtype=float)
            })
    flat_val = []
    for fe in val_entries:
        for ci, ch in enumerate(fe['chunks']):
            flat_val.append({
                "input_ids": ch['input_ids'],
                "attention_mask": ch['attention_mask'],
                "label_id": fe['label_id'],
                "file_path": fe['file_path'],
                "teacher_logits": fe['teacher_logits'][ci] if (fe['teacher_logits'].shape[0] > ci) else np.zeros((num_labels,), dtype=float)
            })
    # build flat_test correctly
    flat_test = []
    for fe in test_entries:
        for ci, ch in enumerate(fe['chunks']):
            flat_test.append({
                "input_ids": ch['input_ids'],
                "attention_mask": ch['attention_mask'],
                "label_id": fe['label_id'],
                "file_path": fe['file_path'],
                "teacher_logits": np.zeros((num_labels,), dtype=float)
            })

    train_dataset = ChunkDataset(flat_train)
    val_dataset = ChunkDataset(flat_val)
    test_dataset = ChunkDataset(flat_test)

    train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS)

    # student & teacher model
    student_config = AutoConfig.from_pretrained(STUDENT_MODEL_NAME, num_labels=num_labels)
    student_model = AutoModelForSequenceClassification.from_pretrained(STUDENT_MODEL_NAME, config=student_config)
    student_model.to(DEVICE)
    teacher_model = AutoModelForSequenceClassification.from_pretrained(str(teacher_model_dir))
    teacher_model.to(DEVICE)
    teacher_model.eval()

    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        {"params": [p for n,p in student_model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n,p in student_model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0}
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE, eps=1e-8)
    total_steps = len(train_loader) * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps)

    ce_loss_fct = nn.CrossEntropyLoss()
    kl_loss_fct = nn.KLDivLoss(reduction="batchmean")

    best_val_macro_f1 = -1.0
    best_epoch = -1
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "val_macro_f1": []}

    print("[TRAIN] Distillation training start on device", DEVICE)
    for epoch in range(1, MAX_EPOCHS+1):
        student_model.train()
        running_loss = 0.0
        nb = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch} train", leave=False):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            student_outputs = student_model(input_ids=input_ids, attention_mask=attention_mask)
            stu_logits = student_outputs.logits  # (B,C)

            # teacher logits for this batch from precomputed tensor
            teacher_logits_tensor = batch.get("teacher_logits")
            if teacher_logits_tensor is None:
                # fallback: compute teacher logits on-the-fly (rare if mapping worked)
                with torch.no_grad():
                    t_out = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
                    t_logits = t_out.logits.detach().cpu().numpy()
                teacher_logits_tensor = torch.tensor(t_logits, dtype=torch.float32).to(DEVICE)
            else:
                teacher_logits_tensor = teacher_logits_tensor.to(DEVICE)

            # KD loss
            T = TEMPERATURE
            student_log_probs = nn.functional.log_softmax(stu_logits / T, dim=-1)
            teacher_probs = nn.functional.softmax(teacher_logits_tensor / T, dim=-1)
            kd_loss = kl_loss_fct(student_log_probs, teacher_probs) * (T * T)
            ce_loss = ce_loss_fct(stu_logits, labels)
            loss = ALPHA * kd_loss + (1.0 - ALPHA) * ce_loss

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student_model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            nb += 1

        avg_train_loss = running_loss / max(1, nb)
        history["train_loss"].append(avg_train_loss)

        # validation
        student_model.eval()
        val_logits_by_file = {fe['file_path']: [] for fe in val_entries}
        val_loss_acc = 0.0
        val_steps = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch} val", leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                out = student_model(input_ids=input_ids, attention_mask=attention_mask)
                logits = out.logits.detach().cpu().numpy()
                for i, fp in enumerate(batch['file_paths']):
                    val_logits_by_file[fp].append(logits[i])
                # approximate CE per batch (logits->torch), used only for trend
                try:
                    batch_logits_t = torch.tensor(logits, dtype=torch.float32)
                    val_loss_acc += float(ce_loss_fct(batch_logits_t, labels.detach().cpu()).item())
                except Exception:
                    pass
                val_steps += 1

        for fp in val_logits_by_file:
            if val_logits_by_file[fp]:
                val_logits_by_file[fp] = np.vstack(val_logits_by_file[fp])
            else:
                val_logits_by_file[fp] = np.zeros((0, num_labels), dtype=float)
        val_agg = aggregate_chunk_logits_to_file(val_logits_by_file)

        y_true = []
        y_pred = []
        for fe in val_entries:
            fp = fe['file_path']
            true_id = int(fe['label_id'])
            logits = val_agg.get(fp, np.zeros((num_labels,), dtype=float))
            pred_id = int(np.argmax(logits)) if logits.size else -1
            y_true.append(true_id)
            y_pred.append(pred_id if pred_id >= 0 else 0)

        val_metrics = compute_metrics_file_level(y_true, y_pred, inv_label_map)
        avg_val_loss = (val_loss_acc / max(1, val_steps)) if val_steps else 0.0
        history["val_loss"].append(avg_val_loss)
        history["val_macro_f1"].append(val_metrics["macro_f1"])

        print(f"[EPOCH {epoch}] train_loss={avg_train_loss:.4f} val_loss={avg_val_loss:.4f} val_macro_f1={val_metrics['macro_f1']:.4f}")

        if val_metrics["macro_f1"] > best_val_macro_f1 + 1e-6:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            epochs_no_improve = 0
            student_model.save_pretrained(MODEL_DIR)
            AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME).save_pretrained(MODEL_DIR)
            json.dump(make_json_serializable({"label_map": label_map, "inv_label_map": {str(k):v for k,v in inv_label_map.items()}, "metadata": metadata}),
                      open(MODEL_DIR / "label_map.json", "w", encoding="utf8"),
                      indent=2, ensure_ascii=False)
            json.dump(make_json_serializable({"epoch": epoch, "val_metrics": val_metrics, "train_loss": avg_train_loss}),
                      open(MODEL_DIR / "best_epoch_metrics.json", "w", encoding="utf8"),
                      indent=2, ensure_ascii=False)
            print(f"[MODEL] Saved best student model at epoch {epoch}")
        else:
            epochs_no_improve += 1
            print(f"[EARLY_STOPPING] no improve {epochs_no_improve}/{EARLY_STOPPING_PATIENCE}")

        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print("[EARLY_STOPPING] reached patience -> stop")
            break

    if not (MODEL_DIR.exists() and any(MODEL_DIR.iterdir())):
        print("[WARN] best student model not saved; saving current model as best")
        student_model.save_pretrained(MODEL_DIR)
        AutoTokenizer.from_pretrained(STUDENT_MODEL_NAME).save_pretrained(MODEL_DIR)
        json.dump(make_json_serializable({"label_map": label_map, "inv_label_map": {str(k):v for k,v in inv_label_map.items()}, "metadata": metadata}),
                  open(MODEL_DIR / "label_map.json", "w", encoding="utf8"),
                  indent=2, ensure_ascii=False)

    # final test evaluation
    student_eval = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR)).to(DEVICE)
    student_eval.eval()
    test_logits_by_file = {fe['file_path']: [] for fe in test_entries}
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Test inference", leave=False):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            out = student_eval(input_ids=input_ids, attention_mask=attention_mask)
            logits = out.logits.detach().cpu().numpy()
            for i, fp in enumerate(batch['file_paths']):
                test_logits_by_file[fp].append(logits[i])

    for fp in test_logits_by_file:
        if test_logits_by_file[fp]:
            test_logits_by_file[fp] = np.vstack(test_logits_by_file[fp])
        else:
            test_logits_by_file[fp] = np.zeros((0, num_labels), dtype=float)
    test_agg = aggregate_chunk_logits_to_file(test_logits_by_file)

    test_rows = []
    y_true = []; y_pred = []
    for fe in test_entries:
        fp = fe['file_path']; true_id = int(fe['label_id'])
        logits = test_agg.get(fp, np.zeros((num_labels,), dtype=float))
        pred_id = int(np.argmax(logits)) if logits.size else -1
        test_rows.append({"file_path": fp, "gold_label_id": true_id, "gold_label_str": inv_label_map.get(true_id, None), "pred_label_id": pred_id, "pred_label_str": inv_label_map.get(pred_id, None)})
        y_true.append(true_id); y_pred.append(pred_id if pred_id>=0 else 0)

    preds_df = pd.DataFrame(test_rows)
    preds_csv = EXP_OUT / "predictions.csv"
    preds_df.to_csv(preds_csv, index=False, encoding="utf8")
    print(f"[EVAL] Saved predictions: {preds_csv}")

    test_metrics = compute_metrics_file_level(y_true, y_pred, inv_label_map)
    metrics_path = EXP_OUT / "metrics.json"
    json.dump(make_json_serializable(test_metrics), open(metrics_path, "w", encoding="utf8"), indent=2, ensure_ascii=False)
    print(f"[EVAL] Saved metrics: {metrics_path}")

    np.savez_compressed(str(LOG_DIR / "student_test_chunk_logits.npz"), **test_logits_by_file)
    print(f"[INFO] Saved student chunk logits to {LOG_DIR / 'student_test_chunk_logits.npz'}")

    summary = {
        "language": LANG,
        "student_model_name": STUDENT_MODEL_NAME,
        "teacher_exp_dir": str(teacher_exp_dir),
        "label_map": label_map,
        "metadata": metadata,
        "training": {
            "max_epochs": MAX_EPOCHS,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "eval_batch_size": EVAL_BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_macro_f1,
            "history": history,
            "distillation": {"temperature": TEMPERATURE, "alpha": ALPHA, "topk_k": TOPK_K}
        },
        "paths": {
            "exp_out": str(EXP_OUT.resolve()),
            "model_dir": str(MODEL_DIR.resolve()),
            "predictions_csv": str(preds_csv.resolve()),
            "metrics_json": str(metrics_path.resolve()),
            "student_logits_npz": str(LOG_DIR.resolve()),
            "topk_info": str(TOPK_INFO_PATH.resolve())
        }
    }
    json.dump(make_json_serializable(summary), open(EXP_OUT / "summary.json", "w", encoding="utf8"), indent=2, ensure_ascii=False)
    print(f"[DONE] Distillation outputs saved under {EXP_OUT.resolve()}")
    print("="*100)

print("\n[ALL DONE] Distillation finished for all languages.")


[INFO] Languages detected: ['English', 'Hindi', 'Marathi']

[LANG] English
[INFO] label_map: {'G': 0, 'PG': 1, 'PG-13': 2, 'R': 3, 'NC-17': 4}  num_labels=5
[INFO] chunk params: max_len=256, stride=64
[INFO] Loaded teacher logits: train_keys=799 val_keys=171
[INFO] Loaded file_path_key_mapping.json (970 entries) from /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/train_pipeline/English/teacher_distilbert-base-multilingual-cased/logits/file_path_key_mapping.json


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

...

[TRAIN] Distillation training start on device cuda


[EPOCH 1] train_loss=0.7491 val_loss=0.6869 val_macro_f1=0.5833
[MODEL] Saved best student model at epoch 1


[EPOCH 2] train_loss=0.6845 val_loss=0.7096 val_macro_f1=0.4976
[EARLY_STOPPING] no improve 1/2


[EPOCH 3] train_loss=0.5753 val_loss=0.8641 val_macro_f1=0.4976
[EARLY_STOPPING] no improve 2/2
[EARLY_STOPPING] reached patience -> stop


[EVAL] Saved predictions: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/distillation/Marathi/distil_distilbert-base-multilingual-cased/predictions.csv
[EVAL] Saved metrics: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/distillation/Marathi/distil_distilbert-base-multilingual-cased/metrics.json
[INFO] Saved student chunk logits to /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/distillation/Marathi/distil_distilbert-base-multilingual-cased/logits/student_test_chunk_logits.npz
[DONE] Distillation outputs saved under /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/distillation/Marathi/distil_distilbert-base-multilingual-cased

[ALL DONE] Distillation finished for all languages.


In [1]:
import json, pandas as pd
from pathlib import Path

for L in ["English","Hindi","Marathi"]:
    lm = Path(f"/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/data_splits/{L}/label_map.json")
    print(L, json.load(open(lm))['label_map'])
    preds = pd.read_csv(f"/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/distillation/{L}/distil_distilbert-base-multilingual-cased/predictions.csv")
    print("preds head:", preds.head())


English {'G': 0, 'PG': 1, 'PG-13': 2, 'R': 3, 'NC-17': 4}
preds head:                                            file_path  gold_label_id  \
0  /content/drive/MyDrive/PhDWorks/4_Final_Writin...              2   
1  /content/drive/MyDrive/PhDWorks/4_Final_Writin...              3   
2  /content/drive/MyDrive/PhDWorks/4_Final_Writin...              2   
3  /content/drive/MyDrive/PhDWorks/4_Final_Writin...              1   
4  /content/drive/MyDrive/PhDWorks/4_Final_Writin...              3   

  gold_label_str  pred_label_id pred_label_str  
0          PG-13              3              R  
1              R              2          PG-13  
2          PG-13              3              R  
3             PG              3              R  
4              R              3              R  
Hindi {'U': 0, 'UA': 1, 'A': 2}
preds head:                                            file_path  gold_label_id  \
0  /content/drive/MyDrive/PhDWorks/4_Final_Writin...              1   
1  /content/drive/MyDri

In [1]:
from pathlib import Path
import json

root = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2")
langs = ["English","Hindi","Marathi"]

for L in langs:
    print(f"\n[CHECK] {L}")
    lm = root / "data_splits" / L / "label_map.json"
    print("label_map exists:", lm.exists())
    print("label_map:", json.load(open(lm))['label_map'])

    train_logits = root / "outputs" / "train_pipeline" / L / "teacher_distilbert-base-multilingual-cased" / "logits" / "teacher_train_chunk_logits.npz"
    print("teacher_train_chunk_logits exists:", train_logits.exists())

    student_preds = root / "outputs" / "distillation" / L / "distil_distilbert-base-multilingual-cased" / "predictions.csv"
    print("student predictions exists:", student_preds.exists())



[CHECK] English
label_map exists: True
label_map: {'G': 0, 'PG': 1, 'PG-13': 2, 'R': 3, 'NC-17': 4}
teacher_train_chunk_logits exists: True
student predictions exists: True

[CHECK] Hindi
label_map exists: True
label_map: {'U': 0, 'UA': 1, 'A': 2}
teacher_train_chunk_logits exists: True
student predictions exists: True

[CHECK] Marathi
label_map exists: True
label_map: {'U': 0, 'UA': 1}
teacher_train_chunk_logits exists: True
student predictions exists: True
